# Semif + Clustering

## Setup:

In [1]:
!git clone https://github.com/bitsnaps/SemIf-OpenJev

Cloning into 'SemIf-OpenJev'...
remote: Enumerating objects: 460, done.
remote: Counting objects: 100% (268/268), done.
remote: Compressing objects: 100% (190/190), done.
remote: Total 460 (delta 149), reused 90 (delta 75), pack-reused 192 (from 1)
Receiving objects: 100% (460/460), 10.88 MiB | 18.73 MiB/s, done.
Resolving deltas: 100% (191/191), done.


In [2]:
%cd SemIf-OpenJev

/content/SemIf-OpenJev


In [ ]:
!pip install .

In [4]:
!ls examples/cluster_records.jsonl

examples/cluster_records.jsonl


In [2]:
# must run to fix some issues on colab
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.1.6
Uninstalling torchvision-0.1.6:
  Successfully uninstalled torchvision-0.1.6
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 750.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90

In [2]:
%cd SemIf-OpenJev

/content/SemIf-OpenJev


## Based on small Qwen3 models:

### Qwen3-0.6B:

In [1]:
from semif_phase1 import profiles, clustering
from semif_phase1.core import load_causal_model
from semif_phase1.shared import score_shared

# Pinned baseline from manifests/models.json (fits a T4 easily)
model, tokenizer, metadata = load_causal_model(
    "Qwen/Qwen3-0.6B",
    revision="c1899de289a04d12100db370d81485cdf75e47ca",
    device="cuda", dtype="bfloat16",
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [4]:
import json

records = [json.loads(l) for l in open("examples/cluster_records.jsonl")]

all_scorings = []
for record in records:                     # {"id": ..., "state": ...}
    rows = profiles.probe_rows(record)     # 5 probe rows sharing one state
    results, _timing = score_shared(       # one prefill per record
        model, tokenizer, rows, metadata, max_tokens=4096)
    all_scorings.extend(results)           # native scorer output now accepted

artifact = profiles.aggregate(all_scorings)
clusters = clustering.assign(artifact, method="hdbscan", min_cluster_size=5)

In [5]:
clusters

{'battery_version': 'semantic-profiles-v1',
 'battery_sha256': 'b56beb732d26656388d468b211d123f9002b704b601375f5eba8037f32b9dfd2',
 'feature_names': ['affirm_mean',
  'deny_mean',
  'insufficient_mean',
  'entropy_mean'],
 'standardization': {'means': [0.8978802727991854,
   0.1018783771274284,
   0.00024135007338622498,
   0.2800696675368578],
  'stds': [0.15473847161387433,
   0.1547901825202058,
   0.0006495978446041716,
   0.18787458682897015]},
 'method': 'HDBSCAN',
 'params': {'min_cluster_size': 5, 'min_samples': None, 'seed': 217},
 'n_records': 40,
 'source_sha256': '3374b7a528a9a37b437ccedbff2ef4f0b2082ceef400822c7bcaec290be18650',
 'clusters': {'cluster-0': {'size': 27, 'representative_record_id': 'urg-01'},
  'cluster-1': {'size': 7, 'representative_record_id': 'refund-07'}},
 'memberships': {'cluster-0': ['bill-01',
   'bill-03',
   'bill-05',
   'bill-06',
   'bill-07',
   'acct-02',
   'acct-03',
   'acct-04',
   'acct-06',
   'acct-07',
   'refund-03',
   'refund-06',
 

In [46]:
i = 0
memberships = clusters['memberships']
for cluster in memberships:
  print(cluster, ': ', memberships[f"cluster-{i}"])
  i += 1

cluster-0 :  ['bill-01', 'bill-03', 'bill-05', 'bill-06', 'bill-07', 'acct-02', 'acct-03', 'acct-04', 'acct-06', 'acct-07', 'refund-03', 'refund-06', 'sec-01', 'sec-02', 'sec-03', 'sec-04', 'sec-06', 'sec-07', 'urg-01', 'urg-02', 'urg-04', 'urg-05', 'urg-06', 'misc-01', 'misc-03', 'misc-04', 'misc-05']
cluster-1 :  ['bill-04', 'acct-05', 'refund-01', 'refund-02', 'refund-04', 'refund-07', 'urg-07']


In [12]:
def find_clusters(records, min_cluster_size=2):
  scores = []
  for record in records:
    rows = profiles.probe_rows(record)
    result, _timing = score_shared(model, tokenizer, rows, metadata, max_tokens=4096)
    scores.extend(result)
  artifact = profiles.aggregate(scores)
  return clustering.assign(artifact, method="hdbscan", min_cluster_size=min_cluster_size)

In [39]:
items = [{"id": "i1", "state": "red"}, {"id": "i2", "state": "blue"}, {"id": "i3", "state": "table"}, {"id": "i4", "state": "chair"}, {"id": "i5", "state": "car"}, {"id": "i6", "state": "boat"}, {"id": "i7", "state": "plan"}]
results = find_clusters(items, min_cluster_size=2)
results

{'battery_version': 'semantic-profiles-v1',
 'battery_sha256': 'b56beb732d26656388d468b211d123f9002b704b601375f5eba8037f32b9dfd2',
 'feature_names': ['affirm_mean',
  'deny_mean',
  'insufficient_mean',
  'entropy_mean'],
 'standardization': {'means': [0.9557179699793894,
   0.04354612360957779,
   0.0007359064110327658,
   0.17597336432650862],
  'stds': [0.07147089286577894,
   0.07102898515857463,
   0.0005136528392569404,
   0.19635038234280713]},
 'method': 'HDBSCAN',
 'params': {'min_cluster_size': 2, 'min_samples': None, 'seed': 217},
 'n_records': 7,
 'source_sha256': 'd04cbefc21cc6611692692601aa7fa593cac97cfe7175e5d79120b9ecb3e4c72',
 'clusters': {'cluster-0': {'size': 3, 'representative_record_id': 'i4'},
  'cluster-1': {'size': 2, 'representative_record_id': 'i3'}},
 'memberships': {'cluster-0': ['i1', 'i4', 'i5'], 'cluster-1': ['i3', 'i7']},
 'noise_record_ids': ['i2', 'i6']}

In [40]:
i = 0
memberships = results['memberships']
for cluster in memberships:
  print(cluster, ': ', memberships[f"cluster-{i}"])
  i += 1

cluster-0 :  ['i1', 'i4', 'i5']
cluster-1 :  ['i3', 'i7']


In [42]:
items = [{"id": "i1", "state": "red"}, {"id": "i2", "state": "blue"}, {"id": "i3", "state": "table"}, {"id": "i4", "state": "chair"}, {"id": "i5", "state": "car"}, {"id": "i6", "state": "boat"}, {"id": "i7", "state": "plan"}]
results = find_clusters(items, min_cluster_size=3)
results

{'battery_version': 'semantic-profiles-v1',
 'battery_sha256': 'b56beb732d26656388d468b211d123f9002b704b601375f5eba8037f32b9dfd2',
 'feature_names': ['affirm_mean',
  'deny_mean',
  'insufficient_mean',
  'entropy_mean'],
 'standardization': {'means': [0.9557179699793894,
   0.04354612360957779,
   0.0007359064110327658,
   0.17597336432650862],
  'stds': [0.07147089286577894,
   0.07102898515857463,
   0.0005136528392569404,
   0.19635038234280713]},
 'method': 'HDBSCAN',
 'params': {'min_cluster_size': 3, 'min_samples': None, 'seed': 217},
 'n_records': 7,
 'source_sha256': 'd04cbefc21cc6611692692601aa7fa593cac97cfe7175e5d79120b9ecb3e4c72',
 'clusters': {},
 'memberships': {},
 'noise_record_ids': ['i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7']}

In [43]:
items = [{"id": "i1", "state": "red"}, {"id": "i2", "state": "blue"}, {"id": "i3", "state": "table"}, {"id": "i4", "state": "chair"}, {"id": "i5", "state": "car"}, {"id": "i6", "state": "boat"}, {"id": "i7", "state": "plan"}]
results = find_clusters(items, min_cluster_size=4)
results

{'battery_version': 'semantic-profiles-v1',
 'battery_sha256': 'b56beb732d26656388d468b211d123f9002b704b601375f5eba8037f32b9dfd2',
 'feature_names': ['affirm_mean',
  'deny_mean',
  'insufficient_mean',
  'entropy_mean'],
 'standardization': {'means': [0.9557179699793894,
   0.04354612360957779,
   0.0007359064110327658,
   0.17597336432650862],
  'stds': [0.07147089286577894,
   0.07102898515857463,
   0.0005136528392569404,
   0.19635038234280713]},
 'method': 'HDBSCAN',
 'params': {'min_cluster_size': 4, 'min_samples': None, 'seed': 217},
 'n_records': 7,
 'source_sha256': 'd04cbefc21cc6611692692601aa7fa593cac97cfe7175e5d79120b9ecb3e4c72',
 'clusters': {},
 'memberships': {},
 'noise_record_ids': ['i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7']}

In [44]:
items = [{"id": "i1", "state": "red"}, {"id": "i2", "state": "blue"}, {"id": "i3", "state": "table"}, {"id": "i4", "state": "chair"}, {"id": "i5", "state": "car"}, {"id": "i6", "state": "boat"}, {"id": "i7", "state": "plan"}]
results = find_clusters(items, min_cluster_size=5)
results

{'battery_version': 'semantic-profiles-v1',
 'battery_sha256': 'b56beb732d26656388d468b211d123f9002b704b601375f5eba8037f32b9dfd2',
 'feature_names': ['affirm_mean',
  'deny_mean',
  'insufficient_mean',
  'entropy_mean'],
 'standardization': {'means': [0.9557179699793894,
   0.04354612360957779,
   0.0007359064110327658,
   0.17597336432650862],
  'stds': [0.07147089286577894,
   0.07102898515857463,
   0.0005136528392569404,
   0.19635038234280713]},
 'method': 'HDBSCAN',
 'params': {'min_cluster_size': 5, 'min_samples': None, 'seed': 217},
 'n_records': 7,
 'source_sha256': 'd04cbefc21cc6611692692601aa7fa593cac97cfe7175e5d79120b9ecb3e4c72',
 'clusters': {},
 'memberships': {},
 'noise_record_ids': ['i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7']}

### Qwen3.5-4B:

In [49]:
from semif_phase1 import profiles, clustering
from semif_phase1.core import load_causal_model
from semif_phase1.shared import score_shared

# Pinned baseline from manifests/models.json (fits a T4 easily)
model, tokenizer, metadata = load_causal_model(
    "Qwen/Qwen3.5-4B",
    revision="851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
    device="cuda", dtype="bfloat16",
)

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [50]:

records = [json.loads(l) for l in open("examples/cluster_records.jsonl")]

all_scorings = []
for record in records:                     # {"id": ..., "state": ...}
    rows = profiles.probe_rows(record)     # 5 probe rows sharing one state
    results, _timing = score_shared(       # one prefill per record
        model, tokenizer, rows, metadata, max_tokens=4096)
    all_scorings.extend(results)           # native scorer output now accepted

artifact = profiles.aggregate(all_scorings)
clusters = clustering.assign(artifact, method="hdbscan", min_cluster_size=5)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


In [51]:
clusters

{'battery_version': 'semantic-profiles-v1',
 'battery_sha256': 'b56beb732d26656388d468b211d123f9002b704b601375f5eba8037f32b9dfd2',
 'feature_names': ['affirm_mean',
  'deny_mean',
  'insufficient_mean',
  'entropy_mean'],
 'standardization': {'means': [0.4178085373617008,
   0.5703299883552725,
   0.011861474283026639,
   0.2644100892490901],
  'stds': [0.18710334629698705,
   0.185524638700061,
   0.012383068363062485,
   0.14832509222598378]},
 'method': 'HDBSCAN',
 'params': {'min_cluster_size': 5, 'min_samples': None, 'seed': 217},
 'n_records': 40,
 'source_sha256': 'a5b65137d474940edddb04db64b2eb8b0c6a5da674b25a51e0700a9704304db4',
 'clusters': {},
 'memberships': {},
 'noise_record_ids': ['bill-01',
  'bill-02',
  'bill-03',
  'bill-04',
  'bill-05',
  'bill-06',
  'bill-07',
  'acct-01',
  'acct-02',
  'acct-03',
  'acct-04',
  'acct-05',
  'acct-06',
  'acct-07',
  'refund-01',
  'refund-02',
  'refund-03',
  'refund-04',
  'refund-05',
  'refund-06',
  'refund-07',
  'sec-01'

## Based on MiniCPM:

In [45]:
import json
from semif_phase1 import profiles, clustering
from semif_phase1.core import load_causal_model
from semif_phase1.shared import score_shared

# Pinned baseline from manifests/models.json (fits a T4 easily)
model, tokenizer, metadata = load_causal_model(
    "openbmb/MiniCPM5-2B",
    revision="12a3808a956f869c767195e9266b59c4d21d92e2",
    device="cuda", dtype="bfloat16",
)

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/94.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.89M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/31.4k [00:00<?, ?B/s]

model-00000-of-00001.safetensors: reconstructing file:   0%|          |  0.00B / 5.03GB            

model-00000-of-00001.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/381 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

In [47]:

records = [json.loads(l) for l in open("examples/cluster_records.jsonl")]

all_scorings = []
for record in records:                     # {"id": ..., "state": ...}
    rows = profiles.probe_rows(record)     # 5 probe rows sharing one state
    results, _timing = score_shared(       # one prefill per record
        model, tokenizer, rows, metadata, max_tokens=4096)
    all_scorings.extend(results)           # native scorer output now accepted

artifact = profiles.aggregate(all_scorings)
clusters = clustering.assign(artifact, method="hdbscan", min_cluster_size=5)

In [48]:
clusters

{'battery_version': 'semantic-profiles-v1',
 'battery_sha256': 'b56beb732d26656388d468b211d123f9002b704b601375f5eba8037f32b9dfd2',
 'feature_names': ['affirm_mean',
  'deny_mean',
  'insufficient_mean',
  'entropy_mean'],
 'standardization': {'means': [0.2031529154934254,
   0.7928275848330056,
   0.004019499673568921,
   0.6013162015286412],
  'stds': [0.07171004793574534,
   0.06892115499123756,
   0.01005689928429473,
   0.1100728067576488]},
 'method': 'HDBSCAN',
 'params': {'min_cluster_size': 5, 'min_samples': None, 'seed': 217},
 'n_records': 40,
 'source_sha256': 'e86823cccb776863488456fcdce7f6d09f7bedfeebea97c8b50ef4d85ff0a6df',
 'clusters': {},
 'memberships': {},
 'noise_record_ids': ['bill-01',
  'bill-02',
  'bill-03',
  'bill-04',
  'bill-05',
  'bill-06',
  'bill-07',
  'acct-01',
  'acct-02',
  'acct-03',
  'acct-04',
  'acct-05',
  'acct-06',
  'acct-07',
  'refund-01',
  'refund-02',
  'refund-03',
  'refund-04',
  'refund-05',
  'refund-06',
  'refund-07',
  'sec-01'